# Causal Inference Analysis of Anki Spaced-Repetition Learning Data

**Author:** Siddharth Pottapatri  
**Data span:** January 2025 – April 2026 (~15 months, 52,000+ reviews)  
**Database:** Anki 2.1 SQLite (`collection.anki2`)

---

## Methodology Overview

| Phase | Method | Question |
|-------|--------|----------|
| 1 | Exploratory Data Analysis | What does my review behaviour look like? |
| 2 | Interrupted Time Series (ITS) | Did behavioural changes shift retention trends? |
| 3 | Propensity Score Matching (PSM) | Does reviewing in the morning *cause* better retention? |
| 4 | Survival Analysis | What predicts a card becoming a leech? |

### Causal Assumptions
- **ITS** assumes no simultaneous confounders at the breakpoint (parallel trends counterfactual).
- **PSM** assumes *no unmeasured confounders* conditional on card difficulty, deck, interval, and recency (strong ignorability). We relax this with Rosenbaum bounds sensitivity analysis.
- **Cox PH** assumes the log-hazard ratio between groups is constant over time (verified via Schoenfeld residuals).

---
## Phase 1 — Data Extraction & Exploratory Analysis

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── publication-quality style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.labelsize': 11,
    'axes.titlesize': 12,
})
PALETTE = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

DB_PATH = r'C:\Users\sid99\AppData\Roaming\Anki2\Pottapatri\collection.anki2'
print('DB path:', DB_PATH)

In [ ]:
# ── Connect and load tables ────────────────────────────────────────────────
# Anki 2.1.45+ uses a custom 'unicase' collation; supplying a no-op keeps
# sqlite3 from raising OperationalError on schema inspection.
con = sqlite3.connect(DB_PATH)
con.create_collation('unicase', lambda a, b: (a > b) - (a < b))

cards  = pd.read_sql('SELECT id, nid, did, type, queue, ivl, factor, reps, lapses FROM cards', con)
revlog = pd.read_sql('SELECT id, cid, ease, ivl, lastIvl, factor, time, type FROM revlog', con)
decks  = pd.read_sql('SELECT id, name FROM decks', con)
con.close()

print(f'cards:  {len(cards):,} rows')
print(f'revlog: {len(revlog):,} rows')
print(f'decks:  {len(decks):,} rows')
decks

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────
# Timestamps
revlog['ts']   = pd.to_datetime(revlog['id'], unit='ms')
revlog['date'] = revlog['ts'].dt.date
revlog['hour'] = revlog['ts'].dt.hour
revlog['dow']  = revlog['ts'].dt.dayofweek   # 0=Mon
revlog['week'] = revlog['ts'].dt.isocalendar().week.astype(int)
revlog['year'] = revlog['ts'].dt.year

# Retention signal: ease > 1 means the card was recalled (1 = Again / lapse)
revlog['retained'] = (revlog['ease'] > 1).astype(int)

# Anki ease factor is stored as 1000× (e.g. 2500 = 2.50×)
revlog['ease_factor'] = revlog['factor'] / 1000
cards['ease_factor']  = cards['factor']  / 1000

# Join deck name onto revlog via cards
card_deck = cards[['id', 'did']].rename(columns={'id': 'cid'})
deck_map  = decks.set_index('id')['name'].to_dict()
card_deck['deck'] = card_deck['did'].map(deck_map)
revlog = revlog.merge(card_deck[['cid', 'deck']], on='cid', how='left')

# Time spent (seconds)
revlog['time_s'] = revlog['time'] / 1000

print(revlog[['ts', 'ease', 'retained', 'ease_factor', 'deck']].head())

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────
date_min = revlog['ts'].min()
date_max = revlog['ts'].max()
span_days = (date_max - date_min).days

print('='*55)
print('DATASET OVERVIEW')
print('='*55)
print(f'  Date range:      {date_min.date()} → {date_max.date()} ({span_days} days)')
print(f'  Total reviews:   {len(revlog):,}')
print(f'  Unique cards:    {revlog["cid"].nunique():,}')
print(f'  Avg reviews/day: {len(revlog)/span_days:.1f}')
print(f'  Overall retention rate: {revlog["retained"].mean()*100:.1f}%')
print(f'  Avg time per card:      {revlog["time_s"].median():.1f}s (median)')
print()
print('Retention by deck:')
print(revlog.groupby('deck')['retained'].agg(['mean', 'count'])
        .rename(columns={'mean': 'retention_rate', 'count': 'reviews'})
        .assign(retention_rate=lambda x: x['retention_rate'].map('{:.1%}'.format)))

In [ ]:
# ── Figure 1: Reviews over time ────────────────────────────────────────────
daily = (revlog.groupby('date')
               .agg(reviews=('id', 'count'), retention=('retained', 'mean'))
               .reset_index())
daily['date'] = pd.to_datetime(daily['date'])
daily['reviews_7d'] = daily['reviews'].rolling(7, center=True).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].fill_between(daily['date'], daily['reviews'], alpha=0.25, color=PALETTE[0])
axes[0].plot(daily['date'], daily['reviews_7d'], color=PALETTE[0], lw=2, label='7-day rolling avg')
axes[0].set_ylabel('Reviews per day')
axes[0].set_title('Daily Review Volume & Retention Rate')
axes[0].legend(frameon=False)

ret_7d = daily['retention'].rolling(7, center=True).mean()
axes[1].plot(daily['date'], ret_7d, color=PALETTE[2], lw=2)
axes[1].axhline(daily['retention'].mean(), ls='--', color='grey', lw=1, label='mean retention')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].set_ylabel('Retention rate (7-day)')
axes[1].set_xlabel('Date')
axes[1].legend(frameon=False)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right')

plt.tight_layout()
plt.savefig('fig1_daily_reviews.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Time-of-day heatmap ─────────────────────────────────────────
heat = revlog.groupby(['dow', 'hour'])['retained'].agg(['mean', 'count']).reset_index()
heat_pivot = heat.pivot(index='dow', columns='hour', values='mean')
count_pivot = heat.pivot(index='dow', columns='hour', values='count')

# Only show hours with at least 10 reviews to avoid noise
heat_pivot_masked = heat_pivot.where(count_pivot >= 10)

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    heat_pivot_masked,
    cmap='RdYlGn', vmin=0.7, vmax=1.0,
    linewidths=0.3, linecolor='white',
    yticklabels=day_labels,
    cbar_kws={'label': 'Retention rate', 'format': '{x:.0%}'},
    ax=ax
)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Day of week')
ax.set_title('Retention Rate by Day-of-Week and Hour')
plt.tight_layout()
plt.savefig('fig2_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: Retention curve by card maturity ────────────────────────────
# 'ivl' in revlog is the interval assigned after this review (days)
# Negative ivl means the card is in learning phase (minutes, not days)
mature = revlog[revlog['ivl'] > 0].copy()
mature['maturity_bin'] = pd.cut(
    mature['ivl'],
    bins=[0, 1, 7, 21, 60, 180, 9999],
    labels=['New\n(<1d)', 'Young\n(1-7d)', 'Established\n(7-21d)',
            'Mature\n(21-60d)', 'Very Mature\n(60-180d)', 'Mastered\n(180d+)']
)
maturity_ret = (
    mature.groupby('maturity_bin', observed=True)['retained']
          .agg(['mean', 'count', 'sem'])
          .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(maturity_ret['maturity_bin'].astype(str), maturity_ret['mean'],
              yerr=1.96 * maturity_ret['sem'], capsize=4,
              color=PALETTE[0], alpha=0.85, edgecolor='white')
ax.axhline(0.9, ls='--', color='grey', lw=1, label='Anki target (90%)')
for bar, (_, row) in zip(bars, maturity_ret.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'n={row["count"]:,}', ha='center', va='bottom', fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Retention rate ± 95% CI')
ax.set_xlabel('Card maturity (interval at review)')
ax.set_title('Retention Rate by Card Maturity')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('fig3_retention_by_maturity.png', bbox_inches='tight')
plt.show()

---
## Phase 2 — Interrupted Time Series Analysis

### Methodology
Interrupted Time Series (ITS) is a quasi-experimental design that exploits a known intervention time to estimate causal effects on longitudinal outcomes. We model:

$$Y_t = \beta_0 + \beta_1 t + \beta_2 D_t + \beta_3 (t - t^*) D_t + \varepsilon_t$$

where $D_t = 1$ if $t \ge t^*$ (post-intervention), $\beta_2$ is the **level change**, and $\beta_3$ is the **slope change** post-intervention.

**Intervention identification:** We detect natural breakpoints by looking for structural shifts in daily review volume (≥2 std deviations) or multi-day review gaps.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson

# ── Prepare daily time series ──────────────────────────────────────────────
ts = daily.copy().sort_values('date').reset_index(drop=True)
ts['t'] = np.arange(len(ts))  # integer time index

# ── Detect intervention: first gap of 3+ consecutive no-review days ────────
ts['prev_date'] = ts['date'].shift(1)
ts['gap_days']  = (ts['date'] - ts['prev_date']).dt.days.fillna(1)

# Find first significant gap
gap_rows = ts[ts['gap_days'] >= 3]
if len(gap_rows) > 0:
    intervention_date = gap_rows.iloc[0]['date']
    t_star = gap_rows.iloc[0]['t']
    print(f'Primary intervention detected: {intervention_date} (gap of {gap_rows.iloc[0]["gap_days"]:.0f} days)')
else:
    # Fallback: use the date with the largest review count change
    ts['review_change'] = ts['reviews'].diff().abs()
    idx = ts['review_change'].idxmax()
    intervention_date = ts.loc[idx, 'date']
    t_star = ts.loc[idx, 't']
    print(f'Fallback intervention point: {intervention_date}')

# Also find second breakpoint: largest single-day volume increase after t_star
post_ts = ts[ts['t'] > t_star].copy()
if len(post_ts) >= 14:
    rollstd = ts['reviews'].rolling(14).std()
    post_ts2 = post_ts[post_ts['t'] > t_star + 14]
    if len(post_ts2) > 0:
        idx2 = (post_ts2['reviews'] - ts['reviews_7d']).abs().idxmax()
        intervention_date2 = ts.loc[idx2, 'date']
        t_star2 = ts.loc[idx2, 't']
        print(f'Secondary breakpoint: {intervention_date2}')
    else:
        intervention_date2, t_star2 = None, None
else:
    intervention_date2, t_star2 = None, None

In [ ]:
# ── Build ITS design matrix ────────────────────────────────────────────────
ts['D']       = (ts['t'] >= t_star).astype(int)
ts['t_post']  = (ts['t'] - t_star) * ts['D']   # slope change term

# Outcome: 7-day rolling retention (smoothed to reduce noise)
ts['ret_smooth'] = ts['retention'].rolling(7, min_periods=1, center=True).mean()

its_data = ts.dropna(subset=['ret_smooth'])

# ── Fit segmented regression ───────────────────────────────────────────────
model_its = smf.ols('ret_smooth ~ t + D + t_post', data=its_data).fit(
    cov_type='HC3'  # heteroscedasticity-robust SEs
)
print(model_its.summary())

dw = durbin_watson(model_its.resid)
print(f'\nDurbin-Watson statistic: {dw:.3f}  (2.0 = no autocorrelation)')

In [ ]:
# ── Figure 4: ITS plot ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# Scatter: observed daily retention
ax.scatter(ts['date'], ts['retention'], alpha=0.15, s=8, color='grey', label='Daily retention')

# Smoothed outcome
ax.plot(ts['date'], ts['ret_smooth'], color='grey', lw=1, alpha=0.6)

# Model fit: pre- and post-intervention
pre  = its_data[its_data['t'] < t_star]
post = its_data[its_data['t'] >= t_star]

ax.plot(ts.loc[pre.index, 'date'], model_its.fittedvalues[pre.index],
        color=PALETTE[0], lw=2.5, label='Pre-intervention trend')
ax.plot(ts.loc[post.index, 'date'], model_its.fittedvalues[post.index],
        color=PALETTE[1], lw=2.5, label='Post-intervention trend')

# Counterfactual (extend pre-intervention slope)
counterfactual = (model_its.params['Intercept'] +
                  model_its.params['t'] * post['t'])
ax.plot(ts.loc[post.index, 'date'], counterfactual,
        ls='--', color=PALETTE[0], lw=1.5, alpha=0.7, label='Counterfactual')

# Intervention line
ax.axvline(pd.Timestamp(intervention_date), color='black', lw=1.5, ls=':')
ax.text(pd.Timestamp(intervention_date), ax.get_ylim()[1]*0.98,
        f' Intervention\n {intervention_date}',
        va='top', fontsize=9)

# Annotate level and slope changes
level_change = model_its.params['D']
slope_change = model_its.params['t_post']
ax.text(0.98, 0.05,
        f'Level change: {level_change:+.3f} (p={model_its.pvalues["D"]:.3f})\n'
        f'Slope change: {slope_change:+.5f} (p={model_its.pvalues["t_post"]:.3f})',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8))

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylabel('Retention rate (7-day rolling)')
ax.set_xlabel('Date')
ax.set_title('Interrupted Time Series: Retention Rate Around Intervention')
ax.legend(frameon=False, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right')
plt.tight_layout()
plt.savefig('fig4_its.png', bbox_inches='tight')
plt.show()

# Interpretation
print(f'\n=== ITS Interpretation ===')
direction_level = 'increase' if level_change > 0 else 'decrease'
direction_slope = 'accelerating' if slope_change > 0 else 'decelerating'
print(f'Level change (beta2): {level_change:+.3f} -> {direction_level} in retention')
print(f'Slope change (beta3): {slope_change:+.6f}/day -> {direction_slope} post-intervention')


---
## Phase 3 — Propensity Score Matching: Morning vs Evening Reviews

### Research Question
Does reviewing in the **morning** (06:00–12:00) *cause* higher retention compared to **evening** (18:00–00:00), after controlling for pre-existing card difficulty, deck, interval length, and review recency?

### Causal Diagram (DAG)
```
Card difficulty ──┐
Deck type        ──┤──→ Time-of-day ──→ Retention
Interval length  ──┤           ↑
Recency          ──┘  Lifestyle confounders (unmeasured)
```

**Identification strategy:** Strong ignorability — treatment (morning vs evening) is independent of potential outcomes conditional on the measured covariates. We test robustness with Rosenbaum bounds.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy import stats

# ── Subset to morning (6-12) and evening (18-24) reviews ──────────────────
psm_df = revlog[
    (revlog['hour'].between(6, 11) | revlog['hour'].between(18, 23)) &
    (revlog['ivl'] != 0)  # exclude in-learning cards with 0 interval
].copy()

psm_df['morning'] = (psm_df['hour'].between(6, 11)).astype(int)

# ── Card-level features as confounders ─────────────────────────────────────
# Interval (absolute value — negative means learning phase in minutes)
psm_df['abs_ivl'] = psm_df['ivl'].abs()
psm_df['log_ivl'] = np.log1p(psm_df['abs_ivl'])

# Deck indicator
psm_df['deck_enc'] = (psm_df['deck'] == 'Human Japanese Intermediate Shared').astype(int)

# Time since last review (proxy for recency): difference between revlog timestamps for same card
psm_df = psm_df.sort_values(['cid', 'ts'])
psm_df['prev_ts'] = psm_df.groupby('cid')['ts'].shift(1)
psm_df['days_since_last'] = (psm_df['ts'] - psm_df['prev_ts']).dt.total_seconds() / 86400
psm_df['log_days_since'] = np.log1p(psm_df['days_since_last'].fillna(psm_df['days_since_last'].median()))

COVARIATES = ['ease_factor', 'log_ivl', 'deck_enc', 'log_days_since']
psm_clean = psm_df.dropna(subset=COVARIATES + ['retained', 'morning']).copy()

print(f'PSM dataset: {len(psm_clean):,} reviews')
print(f'  Morning: {psm_clean["morning"].sum():,}  |  Evening: {(~psm_clean["morning"].astype(bool)).sum():,}')
print(f'  Baseline retention — Morning: {psm_clean[psm_clean["morning"]==1]["retained"].mean():.1%}  '
      f'| Evening: {psm_clean[psm_clean["morning"]==0]["retained"].mean():.1%}')

In [ ]:
# ── Estimate propensity scores ─────────────────────────────────────────────
X = psm_clean[COVARIATES].values
y = psm_clean['morning'].values

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_sc, y)
psm_clean = psm_clean.copy()
psm_clean['ps'] = lr.predict_proba(X_sc)[:, 1]

print(f'Logistic regression AUC proxy — train accuracy: {lr.score(X_sc, y):.3f}')
print('Propensity score summary:')
print(psm_clean.groupby('morning')['ps'].describe().round(3))

In [ ]:
# ── Figure 5: Propensity score overlap (common support) ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for group, label, color in [(1, 'Morning', PALETTE[0]), (0, 'Evening', PALETTE[1])]:
    subset = psm_clean[psm_clean['morning'] == group]['ps']
    axes[0].hist(subset, bins=50, alpha=0.55, color=color, label=label, density=True)
axes[0].set_xlabel('Propensity score')
axes[0].set_ylabel('Density')
axes[0].set_title('Propensity Score Distribution (Before Matching)')
axes[0].legend(frameon=False)

# Log-odds
eps = 1e-6
psm_clean['log_odds'] = np.log((psm_clean['ps'] + eps) / (1 - psm_clean['ps'] + eps))
for group, label, color in [(1, 'Morning', PALETTE[0]), (0, 'Evening', PALETTE[1])]:
    subset = psm_clean[psm_clean['morning'] == group]['log_odds']
    axes[1].hist(subset, bins=50, alpha=0.55, color=color, label=label, density=True)
axes[1].set_xlabel('Log-odds (propensity)')
axes[1].set_ylabel('Density')
axes[1].set_title('Log-Odds Distribution (Before Matching)')
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig('fig5_psm_overlap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 1:1 Nearest-neighbour matching on log-odds ────────────────────────────
treated   = psm_clean[psm_clean['morning'] == 1].copy()
control   = psm_clean[psm_clean['morning'] == 0].copy()

# Caliper: 0.2 × std of log-odds (Rosenbaum & Rubin 1985 rule-of-thumb)
caliper = 0.2 * psm_clean['log_odds'].std()
print(f'Caliper: {caliper:.4f}')

nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
nbrs.fit(control[['log_odds']].values)
distances, indices = nbrs.kneighbors(treated[['log_odds']].values)

# Apply caliper
matched_mask = distances.flatten() <= caliper
treated_matched  = treated.iloc[matched_mask].copy()
control_matched  = control.iloc[indices.flatten()[matched_mask]].copy()

matched = pd.concat([treated_matched.assign(matched_group='morning'),
                     control_matched.assign(matched_group='evening')])

print(f'Matched pairs: {matched_mask.sum():,} (caliper removed {(~matched_mask).sum():,} treated units)')
print(f'Matched retention — Morning: {treated_matched["retained"].mean():.1%}  '
      f'| Evening: {control_matched["retained"].mean():.1%}')

In [ ]:
# ── Covariate balance check ────────────────────────────────────────────────
def standardized_mean_diff(treated_vals, control_vals):
    """Absolute standardized mean difference (SMD). <0.1 is 'good' balance."""
    mean_diff = treated_vals.mean() - control_vals.mean()
    pooled_std = np.sqrt((treated_vals.std()**2 + control_vals.std()**2) / 2)
    return abs(mean_diff / (pooled_std + 1e-9))

balance_rows = []
for cov in COVARIATES:
    smd_pre  = standardized_mean_diff(treated[cov],         control[cov])
    smd_post = standardized_mean_diff(treated_matched[cov], control_matched[cov])
    balance_rows.append({'covariate': cov, 'SMD_before': smd_pre, 'SMD_after': smd_post})

balance_df = pd.DataFrame(balance_rows)
print('Covariate Balance (SMD — target < 0.1):')
print(balance_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(COVARIATES))
ax.bar(x - 0.2, balance_df['SMD_before'], 0.35, label='Before matching', color=PALETTE[1], alpha=0.8)
ax.bar(x + 0.2, balance_df['SMD_after'],  0.35, label='After matching',  color=PALETTE[2], alpha=0.8)
ax.axhline(0.1, ls='--', color='black', lw=1, label='Balance threshold (0.1)')
ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '\n') for c in COVARIATES])
ax.set_ylabel('Absolute Standardised Mean Difference')
ax.set_title('Covariate Balance Before and After PSM')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('fig6_balance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Average Treatment Effect (ATE) ────────────────────────────────────────
morning_ret = treated_matched['retained'].values
evening_ret = control_matched['retained'].values

ate = morning_ret.mean() - evening_ret.mean()
t_stat, p_val = stats.ttest_ind(morning_ret, evening_ret)

# 95% CI via bootstrap
rng = np.random.default_rng(42)
boot_ates = []
for _ in range(5000):
    idx = rng.integers(0, len(morning_ret), len(morning_ret))
    boot_ates.append(morning_ret[idx].mean() - evening_ret[idx].mean())
ci_lo, ci_hi = np.percentile(boot_ates, [2.5, 97.5])

print('='*50)
print('AVERAGE TREATMENT EFFECT — Morning vs Evening')
print('='*50)
print(f'  Morning retention: {morning_ret.mean():.4f}')
print(f'  Evening retention: {evening_ret.mean():.4f}')
print(f'  ATE:               {ate:+.4f}  ({ate*100:+.2f} pp)')
print(f'  95% CI:            [{ci_lo:+.4f}, {ci_hi:+.4f}]')
print(f'  t = {t_stat:.3f}, p = {p_val:.4f}')
print()
if p_val < 0.05:
    direction = 'higher' if ate > 0 else 'lower'
    print(f'  → Morning reviews show statistically significantly {direction} retention.')
else:
    print(f'  → No statistically significant difference (p={p_val:.3f} > 0.05).')

In [ ]:
# ── Sensitivity analysis: Rosenbaum Bounds (Gamma) ────────────────────────
# Tests how strong an unmeasured confounder would need to be to explain the
# observed treatment effect away.

def rosenbaum_sensitivity(y_treated, y_control, gammas):
    """
    Wilcoxon signed-rank test p-value bounds under hidden bias Γ.
    Returns (gamma, p_upper_bound) pairs.
    """
    diffs = y_treated - y_control
    n = len(diffs)
    results = []
    for gamma in gammas:
        # Upper bound on p-value under worst-case unmeasured confounding
        p_treat = gamma / (1 + gamma)  # max treatment probability
        # Expected value and variance of Wilcoxon statistic under H0
        t_plus = np.sum(diffs > 0)  # simplified sign test proxy
        mu    = n * p_treat
        sigma = np.sqrt(n * p_treat * (1 - p_treat))
        z = (t_plus - mu) / (sigma + 1e-9)
        p_upper = 1 - stats.norm.cdf(z)
        results.append((gamma, p_upper))
    return results

gammas = [1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
sens = rosenbaum_sensitivity(morning_ret, evening_ret, gammas)

print('Rosenbaum Sensitivity Analysis')
print(f'{"Gamma":>8}  {"p-value (upper bound)":>22}')
print('-'*34)
for gamma, p in sens:
    sig = '  *' if p < 0.05 else ''
    print(f'{gamma:>8.2f}  {p:>22.4f}{sig}')
print('* p < 0.05')

---
## Phase 4 - Survival Analysis: First-Lapse Prediction

### Operationalisation
A **first lapse** (ease=1, "Again") is the first time a card is forgotten after entering the review queue — the earliest observable signal of a difficult card.

**Note:** The classic Anki leech threshold is lapses >= 8, but only 1 card has reached it (the dataset is ~15 months old). We therefore model **time to first lapse**, a far more common event (2.6% of cards) that still captures card difficulty meaningfully.

### Survival Setup
- **Event:** card receives its first lapse (ease = Again at least once)
- **Time:** number of reviews until first lapse (right-censored at total reps)
- **Stratification:** ease factor quartile (Q1 = hardest, Q4 = easiest)
- **Key question:** Do cards with lower ease factor lapse sooner?

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# ── Reframe: 'first lapse' event (lapses >= 1) ────────────────────────────
# With only 1 true leech (lapses >= 8) in this dataset, we use the first lapse
# as the event of interest. This answers: what predicts a card being forgotten
# at least once? We stratify by ease factor quartile (the only varying covariate
# since all cards belong to one deck).
LAPSE_THRESHOLD = 1

survival = cards.copy()
survival['event']    = (survival['lapses'] >= LAPSE_THRESHOLD).astype(int)
survival['duration'] = survival['reps'].clip(lower=1)

# Exclude cards with 0 ease factor (never reviewed)
surv_fit = survival[survival['reps'] >= 1].copy()
surv_fit['ease_factor'] = surv_fit['ease_factor'].replace(0, float('nan'))
surv_fit = surv_fit.dropna(subset=['ease_factor'])

# Stratify by ease factor quartile
surv_fit['ef_quartile'] = pd.qcut(
    surv_fit['ease_factor'], q=4,
    labels=['Q1 (Hardest)', 'Q2', 'Q3', 'Q4 (Easiest)']
)

print(f'Survival dataset: {len(surv_fit):,} cards')
print(f'First-lapse events: {surv_fit["event"].sum():,} ({surv_fit["event"].mean():.1%})')
print()
print('Events by ease factor quartile:')
print(surv_fit.groupby('ef_quartile', observed=True)['event'].agg(['sum', 'count', 'mean'])
        .rename(columns={'sum': 'lapses', 'count': 'total', 'mean': 'rate'}))

In [ ]:
# ── Figure 7: Kaplan-Meier survival curves by ease factor quartile ────────
fig, ax = plt.subplots(figsize=(10, 6))

for i, (quartile, group) in enumerate(surv_fit.groupby('ef_quartile', observed=True)):
    kmf = KaplanMeierFitter()
    kmf.fit(
        durations=group['duration'],
        event_observed=group['event'],
        label=str(quartile)
    )
    kmf.plot_survival_function(
        ax=ax, ci_show=True,
        color=PALETTE[i % len(PALETTE)],
        ci_alpha=0.12
    )

# Log-rank test: Q1 (hardest) vs Q4 (easiest)
g1 = surv_fit[surv_fit['ef_quartile'] == 'Q1 (Hardest)']
g4 = surv_fit[surv_fit['ef_quartile'] == 'Q4 (Easiest)']
lr_result = logrank_test(g1['duration'], g4['duration'],
                          g1['event'], g4['event'])
ax.text(0.98, 0.98,
        f'Log-rank test (Q1 vs Q4)\np = {lr_result.p_value:.4f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('Number of reviews (time at risk)')
ax.set_ylabel('Probability of no first lapse')
ax.set_title('Kaplan-Meier Curves: Time to First Lapse by Ease Factor Quartile')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('fig7_km_survival.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Cox Proportional Hazards Model ────────────────────────────────────────
# Ease factor is the primary covariate of interest (only meaningful varying
# covariate in this single-deck, single-type dataset).
cox_df = surv_fit[['duration', 'event', 'ease_factor']].copy()
cox_df['ease_factor_c'] = cox_df['ease_factor'] - cox_df['ease_factor'].mean()

cph = CoxPHFitter(penalizer=0.1)
cph.fit(
    cox_df[['duration', 'event', 'ease_factor_c']],
    duration_col='duration',
    event_col='event'
)
cph.print_summary()

In [ ]:
# ── Figure 8: Hazard ratio forest plot ────────────────────────────────────
hr_df = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
hr_df.columns = ['HR', 'CI_lo', 'CI_hi', 'p']
hr_df = hr_df.reset_index().rename(columns={'index': 'covariate'})
hr_df['label'] = hr_df['covariate'].map({'ease_factor_c': 'Ease factor (mean-centered, +1 unit)'}).fillna(hr_df['covariate'])

fig, ax = plt.subplots(figsize=(7, 3))
y_pos = list(range(len(hr_df)))
colors = [PALETTE[1] if row['p'] < 0.05 else 'grey' for _, row in hr_df.iterrows()]

ax.scatter(hr_df['HR'], y_pos, zorder=3, s=100, color=colors)
for i, row in hr_df.iterrows():
    ax.plot([row['CI_lo'], row['CI_hi']], [i, i], color=colors[i], lw=2)

ax.axvline(1.0, ls='--', color='black', lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(hr_df['label'].tolist())
ax.set_xlabel('Hazard Ratio (95% CI)')
ax.set_title('Cox PH Model: Hazard Ratio for First Lapse')

for i, row in hr_df.iterrows():
    ax.text(max(row['CI_hi'], 1.0) + 0.01, i,
            f'HR={row["HR"]:.3f}  (p={row["p"]:.3f})',
            va='center', fontsize=9, color=colors[i])

ax.set_xlim(0, max(hr_df['CI_hi'].max() * 1.6, 2.0))
plt.tight_layout()
plt.savefig('fig8_cox_hr.png', bbox_inches='tight')
plt.show()

hr_val = hr_df.iloc[0]['HR']
p_cox = hr_df.iloc[0]['p']
direction = 'decreases' if hr_val < 1 else 'increases'
print(f'For each 1-unit increase in ease factor,')
print(f'the hazard of first lapse {direction} by factor {hr_val:.3f} (p={p_cox:.4f})')

In [ ]:
# ── Cox PH assumption check: Schoenfeld residuals ─────────────────────────
print('Testing proportional hazards assumption (Schoenfeld residuals):')
cph.check_assumptions(
    cox_df[['duration', 'event', 'ease_factor_c']],
    p_value_threshold=0.05,
    show_plots=True
)
plt.tight_layout()
plt.savefig('fig9_schoenfeld.png', bbox_inches='tight')
plt.show()

---
## Summary of Findings

| Analysis | Key Finding |
|----------|-------------|
| **EDA** | ~52k reviews over 15 months; overall retention rate reported above. Mature cards show highest retention; new cards show the most variance. |
| **ITS** | A level change (β₂) and slope change (β₃) are estimated at the detected intervention breakpoint. Significance depends on actual data — see Phase 2 output above. |
| **PSM** | After matching on card difficulty, deck, interval, and recency, the morning–evening ATE is reported with bootstrap 95% CI. Rosenbaum sensitivity shows the Γ at which significance breaks down. |
| **Survival** | Kaplan-Meier curves differentiate leech risk across decks (log-rank p reported). Cox PH hazard ratios quantify per-covariate risk of reaching leech threshold. |

### Limitations
1. **Unobserved confounding** in PSM: fatigue, study session length, and sleep quality are not measured.
2. **Single-deck data** limits generalisability. Results may not transfer across subject domains.
3. **ITS assumes no simultaneous confounders** at the breakpoint — life events co-occurring with detected gaps cannot be ruled out.
4. **Survival proxy**: using `reps` as time-to-event rather than calendar time smooths over irregular review spacing.